In [2]:
import pandas as pd 
import numpy as np 
import requests 
import os 
from pathlib import Path

In [9]:
cwd = os.getcwd()
data_path = Path.cwd().parent / "data" 

In [10]:
work_bank_df = pd.read_csv(f"{data_path}\world-bank-income-groups.csv")
work_bank_df.head()

,Entity,Code,Year,World Bank's income classification
0,Afghanistan,AFG,1987,Low-income countries
1,Afghanistan,AFG,1988,Low-income countries
2,Afghanistan,AFG,1989,Low-income countries
3,Afghanistan,AFG,1990,Low-income countries
4,Afghanistan,AFG,1991,Low-income countries


In [58]:
fatalities = pd.read_csv(f"{data_path}\\number_of_reported_fatalities_by_country-year_as-of-03Apr2026.csv")
fatalities

Table 1
COUNTRY     YEAR FATALITIES NaN      NaN
Afghanistan 2017 36360      NaN      NaN
            2018 42991      NaN      NaN
            2019 41419      NaN      NaN
            2020 31037      NaN      NaN
...                                  ...
eSwatini    2022 33         NaN      NaN
            2023 32         NaN      NaN
            2024 10         NaN      NaN
            2025 2          NaN      NaN
            2026 0          NaN      NaN

[2944 rows x 1 columns]

In [59]:
fatalities = fatalities.reset_index()
fatalities.columns = fatalities.iloc[0].values.tolist()
fatalities = fatalities.drop(0).dropna(axis=1)
fatalities

,COUNTRY,YEAR,FATALITIES
1,Afghanistan,2017,36360
2,Afghanistan,2018,42991
3,Afghanistan,2019,41419
4,Afghanistan,2020,31037
5,Afghanistan,2021,42425
...,...,...,...
2939,eSwatini,2022,33
2940,eSwatini,2023,32
2941,eSwatini,2024,10
2942,eSwatini,2025,2


In [60]:
fatalities.to_csv(f"{data_path}\cleaned_fatalities.csv")

In [63]:
world_bank_countries = work_bank_df['Entity'].unique().tolist()
fatalities_countries = fatalities['COUNTRY'].unique().tolist()
income_group = work_bank_df["World Bank's income classification"].unique().tolist()

global_south = work_bank_df.loc[(work_bank_df["World Bank's income classification"]=='Low-income countries') |
                                (work_bank_df["World Bank's income classification"]=='Lower-middle-income countries'), \
                                     'Entity'].unique().tolist()

western = work_bank_df.loc[(work_bank_df["World Bank's income classification"]=='High-income countries'),
                           'Entity'].unique().tolist()

Same countries (count): 209
World Bank only (count): 17
Fatalities only (count): 36

World Bank only: ['Channel Islands', 'Congo', "Cote d'Ivoire", 'Czechia', 'Czechoslovakia', 'Eswatini', 'French Southern Territories', 'Hong Kong', 'Macao', 'Micronesia (country)', 'Netherlands Antilles', 'Saint Martin (French part)', 'Serbia and Montenegro', 'Sint Maarten (Dutch part)', 'USSR', 'United States Virgin Islands', 'Yugoslavia']
Fatalities only: ['Akrotiri and Dhekelia', 'Anguilla', 'Antarctica', 'Bailiwick of Guernsey', 'Bailiwick of Jersey', 'British Indian Ocean Territory', 'Caribbean Netherlands', 'Christmas Island', 'Cocos (Keeling) Islands', 'Cook Islands', 'Czech Republic', 'Falkland Islands', 'French Southern and Antarctic Lands', 'Guadeloupe', 'Heard Island and McDonald Islands', 'Ivory Coast', 'Martinique', 'Micronesia', 'Montserrat', 'Niue', 'Norfolk Island', 'Pitcairn', 'Republic of Congo', 'Reunion', 'Saint Helena, Ascension and Tristan da Cunha', 'Saint Pierre and Miquelon', '

Same countries (count): 209
World Bank only (count): 17
Fatalities only (count): 36

World Bank only: ['Channel Islands', 'Congo', "Cote d'Ivoire", 'Czechia', 'Czechoslovakia', 'Eswatini', 'French Southern Territories', 'Hong Kong', 'Macao', 'Micronesia (country)', 'Netherlands Antilles', 'Saint Martin (French part)', 'Serbia and Montenegro', 'Sint Maarten (Dutch part)', 'USSR', 'United States Virgin Islands', 'Yugoslavia']
Fatalities only: ['Akrotiri and Dhekelia', 'Anguilla', 'Antarctica', 'Bailiwick of Guernsey', 'Bailiwick of Jersey', 'British Indian Ocean Territory', 'Caribbean Netherlands', 'Christmas Island', 'Cocos (Keeling) Islands', 'Cook Islands', 'Czech Republic', 'Falkland Islands', 'French Southern and Antarctic Lands', 'Guadeloupe', 'Heard Island and McDonald Islands', 'Ivory Coast', 'Martinique', 'Micronesia', 'Montserrat', 'Niue', 'Norfolk Island', 'Pitcairn', 'Republic of Congo', 'Reunion', 'Saint Helena, Ascension and Tristan da Cunha', 'Saint Pierre and Miquelon', '

[('Congo', 'Republic of Congo'), ('Eswatini', 'eSwatini')]

In [81]:
similar_country_pairs

[('congo', ['Congo'], ['Republic of Congo']),
 ('eswatini', ['Eswatini'], ['eSwatini'])]

## Country Name Mismatches Between World Bank and Fatalities Datasets

The following countries appear under different names in each dataset but refer to the same country.

In [83]:
# Countries that exist in both datasets but under different names
# Format: fatalities_name -> world_bank_name
country_name_mapping = {
    'Republic of Congo':                      'Congo',
    'Democratic Republic of Congo':                      'Congo',
    'Czech Republic':                         'Czechia',
    'Ivory Coast':                            "Cote d'Ivoire",
    'eSwatini':                               'Eswatini',
    'Micronesia':                             'Micronesia (country)',
    'French Southern and Antarctic Lands':    'French Southern Territories',
    'Saint-Martin':                           'Saint Martin (French part)',
    'Sint Maarten':                           'Sint Maarten (Dutch part)',
    'Virgin Islands, U.S.':                   'United States Virgin Islands',
}

mapping_df = pd.DataFrame([
    {'Fatalities Name': fat, 'World Bank Name': wb}
    for fat, wb in country_name_mapping.items()
])
mapping_df

,Fatalities Name,World Bank Name
0,Republic of Congo,Congo
1,Democratic Republic of Congo,Congo
2,Czech Republic,Czechia
3,Ivory Coast,Cote d'Ivoire
4,eSwatini,Eswatini
5,Micronesia,Micronesia (country)
6,French Southern and Antarctic Lands,French Southern Territories
7,Saint-Martin,Saint Martin (French part)
8,Sint Maarten,Sint Maarten (Dutch part)
9,"Virgin Islands, U.S.",United States Virgin Islands


In [85]:
fatalities['COUNTRY'] = fatalities['COUNTRY'].replace(country_name_mapping)
fatalities[fatalities['COUNTRY'].str.contains('Congo')]

,COUNTRY,YEAR,FATALITIES
658,Congo,1997,2291
659,Congo,1998,3376
660,Congo,1999,7040
661,Congo,2000,1680
662,Congo,2001,2379
663,Congo,2002,6077
664,Congo,2003,3405
665,Congo,2004,400
666,Congo,2005,565
667,Congo,2006,412
